In [11]:
# 统计日志文件中的 QA 验证结果


import re
import os
import glob

# ================= 配置区 =================
log_file_path = 'eval_traj_20260325_004512.log'  # 请替换为你的实际 log 文件路径
traj_dir = 'eval_traj/CS_eval_traj/SFT_IterScholar/' # 轨迹文件所在的目录
# ==========================================

# 统计数据结构
stats = {
    'yes': {'count': 0, 'total_steps': 0, 'details': []},
    'no': {'count': 0, 'total_steps': 0, 'details': []}
}

# 正则提取任务 ID 和 QA 验证结果
# 匹配示例: ✅ 任务 [8] 完成！ (QA 验证: yes)
pattern = re.compile(r'任务\s*\[(\d+)\]\s*完成！.*?\(QA 验证:\s*(yes|no)\)', re.IGNORECASE)

try:
    with open(log_file_path, 'r', encoding='utf-8') as log_file:
        for line in log_file:
            match = pattern.search(line)
            if match:
                task_id = match.group(1)
                qa_result = match.group(2).lower()
                
                # 拼接并查找对应的 jsonl 轨迹文件 
                # 使用 * 忽略前面的时间戳，直接匹配 task_id
                search_pattern = os.path.join(traj_dir, f'*_CS_eval_traj_task{task_id}.jsonl')
                matched_files = glob.glob(search_pattern)
                
                steps = 0
                if matched_files:
                    # 假设每个任务只对应一个文件，取第一个匹配项
                    traj_file = matched_files[0]
                    try:
                        # 统计 JSONL 文件的行数（即步数）
                        with open(traj_file, 'r', encoding='utf-8') as tf:
                            steps = sum(1 for _ in tf)
                    except Exception as e:
                        print(f"⚠️ 无法读取轨迹文件 {traj_file}: {e}")
                else:
                    print(f"⚠️ 警告: 找不到任务 [{task_id}] 对应的轨迹文件！")
                
                # 记录统计数据
                stats[qa_result]['count'] += 1
                stats[qa_result]['total_steps'] += steps
                stats[qa_result]['details'].append((task_id, steps))

    # 输出统计报告
    print("\n📊 =============== 统计报告 =============== 📊")
    for res in ['yes', 'no']:
        count = stats[res]['count']
        total_steps = stats[res]['total_steps']
        avg_steps = (total_steps / count) if count > 0 else 0
        
        icon = "✅" if res == 'yes' else "❌"
        print(f"{icon} [QA 验证: {res.upper()}]")
        print(f"  - 任务数量: {count} 个")
        print(f"  - 消耗总步数: {total_steps} 步")
        print(f"  - 平均步数/任务: {avg_steps:.2f} 步")
        print("-" * 40)

except FileNotFoundError:
    print(f"⚠️ 找不到日志文件: {log_file_path}，请检查路径。")
except Exception as e:
    print(f"⚠️ 发生错误: {e}")


📊 =============== 统计报告 =============== 📊
✅ [QA 验证: YES]
  - 任务数量: 17 个
  - 消耗总步数: 258 步
  - 平均步数/任务: 15.18 步
----------------------------------------
❌ [QA 验证: NO]
  - 任务数量: 33 个
  - 消耗总步数: 1640 步
  - 平均步数/任务: 49.70 步
----------------------------------------


In [9]:
import os
import shutil

file_path = r"D:\DissertationProject\DissertationProject\eval\eval_traj\CS_eval_traj\basemodel_IterScholar\eval_results.json"
backup_path = file_path + '.bak'

# 如果备份文件存在，恢复它
if os.path.exists(backup_path):
    shutil.copy2(backup_path, file_path)
    print(f"✅ 已恢复到原始 JSONL 格式")
    print(f"📁 恢复文件：{file_path}")
else:
    print("❌ 未找到备份文件，无法恢复")


❌ 未找到备份文件，无法恢复
